# LifeSnaps MongoDB Data Load

이 노트북은 CSV가 아니라 로컬 MongoDB에 복원된 LifeSnaps 원본 데이터에서 필요한 데이터를 직접 불러오는 첫 단계입니다.

주의: `fitbit` 컬렉션은 약 7천만 건 이상이므로 전체를 한 번에 DataFrame으로 읽지 않습니다. 먼저 컬렉션 구조를 확인하고, 필요한 `type`만 필터링해서 가져옵니다.

In [3]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint
from typing import Any

import pandas as pd
from pymongo import MongoClient

# pandas 출력 옵션입니다.
# 컬럼이 많은 sleep table을 확인할 때 중간에 생략되지 않도록 넉넉하게 둡니다.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# 노트북 위치: C:\\workSpace\\DeepLearnin_sleep\\notebooks
# 프로젝트 루트: C:\\workSpace\\DeepLearnin_sleep
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# 로컬 MongoDB 접속 정보입니다.
# 이 노트북을 실행하기 전에 mongod가 127.0.0.1:27017에서 실행 중이어야 합니다.
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "rais_anonymized"
FITBIT_COLLECTION = "fitbit"

PROJECT_ROOT

WindowsPath('c:/workSpace/DeepLearnin_sleep')

## 1. MongoDB 연결

`MongoClient`로 로컬 MongoDB에 연결하고, `ping` 명령으로 실제 연결 가능 여부를 먼저 확인합니다.

In [4]:
# serverSelectionTimeoutMS를 짧게 설정하면 MongoDB가 꺼져 있을 때 오래 멈춰 있지 않습니다.
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)

# ping이 성공하면 MongoDB 서버에 정상적으로 연결된 것입니다.
# 실패하면 mongod 실행 여부와 포트(27017)를 먼저 확인해야 합니다.
client.admin.command("ping")

db = client[DB_NAME]
fitbit = db[FITBIT_COLLECTION]

print("MongoDB connected")
print("Database:", DB_NAME)
print("Collections:", db.list_collection_names())

MongoDB connected
Database: rais_anonymized
Collections: ['surveys', 'sema', 'fitbit']


## 2. 원본 컬렉션 크기 확인

여기서는 각 컬렉션의 문서 수만 확인합니다. 특히 `fitbit`은 매우 크기 때문에 전체 데이터를 읽으면 안 됩니다.

In [5]:
for collection_name in ["fitbit", "sema", "surveys"]:
    collection = db[collection_name]

    # estimated_document_count()는 빠른 개수 확인용입니다.
    # 정확한 조건부 count가 필요할 때는 count_documents(query)를 사용합니다.
    count = collection.estimated_document_count()
    print(f"{collection_name}: {count:,} documents")

fitbit: 71,284,346 documents
sema: 15,380 documents
surveys: 935 documents


## 3. Fitbit 문서 구조 샘플 확인

`fitbit` 컬렉션은 `type` 필드로 데이터 종류가 나뉩니다. 먼저 문서 1개만 가져와 top-level 구조를 봅니다.

In [6]:
def compact_value(value: Any, depth: int = 0) -> Any:
    """큰 MongoDB 문서를 노트북에서 보기 좋은 크기로 줄입니다."""

    # 너무 깊은 중첩은 타입 이름만 표시합니다.
    # sleep의 levels.data 같은 긴 리스트가 그대로 출력되는 것을 막기 위함입니다.
    if depth >= 3:
        return type(value).__name__

    if isinstance(value, dict):
        # dict는 앞쪽 key 일부만 확인합니다.
        return {key: compact_value(item, depth + 1) for key, item in list(value.items())[:12]}

    if isinstance(value, list):
        # 리스트는 첫 원소 구조와 길이만 보여줍니다.
        if not value:
            return []
        return [compact_value(value[0], depth + 1), f"... {len(value)} item(s)"]

    return value


# find_one()은 문서 하나만 읽습니다.
# 전체 fitbit 컬렉션을 DataFrame으로 변환하지 않는 것이 중요합니다.
sample_doc = fitbit.find_one()

print("Top-level keys:", sorted(sample_doc.keys()))
pprint(compact_value(sample_doc), width=120)

Top-level keys: ['_id', 'data', 'id', 'type']
{'_id': ObjectId('62cc1f9ab41dcd4b1beae820'),
 'data': {'Infrared to Red Signal Ratio': -4, 'timestamp': '05/24/21 01:03:30'},
 'id': ObjectId('621e2e8e67b776a24055b564'),
 'type': 'estimated_oxygen_variation'}


## 4. Fitbit `type` 목록 확인

모델링에 필요한 데이터를 고르기 위해 `type`별 문서 수를 확인합니다. 이 집계는 큰 컬렉션에서 실행되므로 시간이 조금 걸릴 수 있습니다.

In [7]:
# MongoDB aggregation pipeline입니다.
# 1단계: type별로 그룹화해서 문서 수를 셉니다.
# 2단계: 문서 수가 많은 type부터 정렬합니다.
type_count_pipeline = [
    {"$group": {"_id": "$type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
]

# allowDiskUse=True는 큰 집계에서 MongoDB가 임시 디스크를 사용할 수 있게 합니다.
# maxTimeMS는 무한정 기다리지 않도록 실행 시간 상한을 둡니다.
type_counts = list(fitbit.aggregate(type_count_pipeline, allowDiskUse=True, maxTimeMS=120_000))

type_counts_df = pd.DataFrame(type_counts).rename(columns={"_id": "type"})
type_counts_df

KeyboardInterrupt: 

## 5. MongoDB에서 sleep 데이터 불러오기

수면 예측의 기준 target을 만들기 위해 `type == "sleep"` 문서만 가져옵니다. 이 단계에서도 필요한 필드만 projection으로 제한합니다.

In [ ]:
# sleep 문서의 data 최상위에서 바로 꺼낼 기본 필드입니다.
# efficiency, minutesAsleep, minutesAwake는 target 또는 target 후보로 사용할 수 있습니다.
SLEEP_BASE_FIELDS = [
    "logId",
    "dateOfSleep",
    "startTime",
    "endTime",
    "duration",
    "minutesToFallAsleep",
    "minutesAsleep",
    "minutesAwake",
    "minutesAfterWakeup",
    "timeInBed",
    "efficiency",
    "type",
    "infoCode",
    "mainSleep",
]


def get_nested(data: dict[str, Any], path: tuple[str, ...]) -> Any:
    """중첩 dict에서 값을 안전하게 가져옵니다.

    Fitbit sleep stage 정보는 levels.summary.deep.minutes처럼 깊게 들어 있습니다.
    모든 문서가 같은 stage 구조를 갖는 것은 아니므로, 없는 key는 None으로 둡니다.
    """

    current: Any = data
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    return current


def flatten_sleep_document(document: dict[str, Any]) -> dict[str, Any]:
    """MongoDB sleep 문서 1개를 pandas DataFrame의 행 1개로 변환합니다."""

    data = document.get("data", {})

    # ObjectId는 pandas/CSV에서 다루기 편하도록 문자열로 변환합니다.
    # participant_object_id는 stress, HRV, steps 등 다른 Fitbit type과 join할 때 사용할 key입니다.
    row: dict[str, Any] = {
        "mongo_doc_id": str(document.get("_id")),
        "participant_object_id": str(document.get("id")),
    }

    # 기본 sleep 지표를 복사합니다.
    for field in SLEEP_BASE_FIELDS:
        row[field] = data.get(field)

    # Fitbit sleep stage는 크게 두 체계가 섞일 수 있습니다.
    # stages: deep, light, rem, wake
    # classic: asleep, restless, awake
    # 일단 모두 컬럼으로 만든 뒤, 결측 패턴을 보고 사용할 컬럼을 고릅니다.
    for stage in ("deep", "light", "rem", "wake", "asleep", "restless", "awake"):
        row[f"{stage}_count"] = get_nested(data, ("levels", "summary", stage, "count"))
        row[f"{stage}_minutes"] = get_nested(data, ("levels", "summary", stage, "minutes"))
        row[f"{stage}_30day_avg_minutes"] = get_nested(
            data, ("levels", "summary", stage, "thirtyDayAvgMinutes")
        )

    return row


def load_sleep_from_mongodb(limit: int | None = None) -> pd.DataFrame:
    """MongoDB fitbit 컬렉션에서 sleep 문서만 읽어 DataFrame으로 반환합니다.

    limit:
        None이면 전체 sleep 문서를 읽습니다. 현재 sleep 문서는 약 4천 건이라 전체 로드해도 괜찮습니다.
        빠른 테스트만 하고 싶으면 10, 100처럼 숫자를 넣으면 됩니다.
    """

    # query는 sleep 타입만 선택합니다.
    query = {"type": "sleep"}

    # projection은 MongoDB에서 가져올 필드를 제한합니다.
    # 큰 컬렉션에서는 필요한 필드만 가져오는 습관이 중요합니다.
    projection = {"_id": 1, "id": 1, "data": 1}

    cursor = fitbit.find(query, projection, batch_size=500)
    if limit is not None:
        cursor = cursor.limit(limit)

    sleep_df = pd.DataFrame(flatten_sleep_document(doc) for doc in cursor)

    if sleep_df.empty:
        return sleep_df

    # 날짜/시간 컬럼을 datetime으로 변환합니다.
    # 파싱 실패 값은 NaT가 되며, 이후 결측치 점검 대상이 됩니다.
    for column in ["dateOfSleep", "startTime", "endTime"]:
        sleep_df[column] = pd.to_datetime(sleep_df[column], errors="coerce")

    # 같은 참가자의 수면 기록이 날짜순으로 보이도록 정렬합니다.
    sleep_df = sleep_df.sort_values(["participant_object_id", "dateOfSleep", "startTime"])
    sleep_df = sleep_df.reset_index(drop=True)

    return sleep_df

In [ ]:
# 실제 MongoDB 원본에서 sleep 문서를 불러옵니다.
# sleep 타입은 약 4천 건이므로 전체 로드해도 부담이 작습니다.
sleep_df = load_sleep_from_mongodb()

print("Rows:", len(sleep_df))
print("Columns:", len(sleep_df.columns))
print("Participants:", sleep_df["participant_object_id"].nunique())
print("Date range:", sleep_df["dateOfSleep"].min(), "to", sleep_df["dateOfSleep"].max())

sleep_df.head()

## 6. 불러온 sleep 데이터를 raw CSV로 저장

이 파일은 MongoDB 원본에서 가져온 1차 추출본입니다. 아직 전처리 완료본이 아니므로 `data/raw` 아래에 저장합니다.

In [ ]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

sleep_output_path = RAW_DATA_DIR / "sleep_from_mongodb.csv"
sleep_df.to_csv(sleep_output_path, index=False, encoding="utf-8-sig")

sleep_output_path